# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset](https://doi.org/10.71728/senscience.qs2f-h81p)
using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All data entities (record sets, fields, etc.) are referenced by their `@id`, following best FAIR data practices.

### Dataset Source
The dataset schema is provided via Croissant (JSON-LD) at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata (does not download data files yet)
dataset = mlc.Dataset(croissant_url)

# Print overview of dataset metadata (using the API, not as dict)
meta = dataset.metadata
print(f"Dataset: {meta.name}\n\nDescription: {meta.description}\n")
print(f"Identifier: {meta.identifier}    Version: {meta.version}")
print(f"License: {meta.license}\nAuthors: {[a for a in getattr(meta, 'author', [])]}")

## 2. Data Overview

Let's inspect the available **record sets** defined by the Croissant schema and list their IDs and fields.

All entity IDs are taken from the schema's `@id` fields.

In [ ]:
# List all RecordSets with their @id, name, and field ids
record_sets = list(dataset.record_sets)
print(f"Record sets found: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '(No name)'}")
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for f in rs.fields:
            print(f"    - @id: {f.id}  name: {f.name}")
    else:
        print("  (No fields)")
    print()
    record_set_ids.append(rs.id)
if not record_sets:
    print('No record sets defined in the Croissant metadata!')

## 3. Data Extraction

We now load the records for the available record set(s). All `@id`s are referenced explicitly. Records are loaded into pandas DataFrames keyed by record set `@id`.

In [ ]:
# Extract records for each record set (@id)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f" Loaded {len(df)} rows. Columns (@id): {list(df.columns)}\n")
        else:
            print(" No data rows loaded.\n")
    except Exception as e:
        print(f" Failed loading data for {record_set_id}: {e}\n")

if dataframes:
    # For demonstration, pick the first available record set for further EDA
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Example record set chosen for EDA: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded. Please check dataset availability.')

## 4. Exploratory Data Analysis (EDA)

Apply basic processing on a numeric field (by column `@id`), filter records, normalize, and group by another field. All IDs below use the actual column `@id`s printed above.

_Replace the field IDs below if your dataset uses different field names!_

In [ ]:
# Pick a numeric and a group field @id from the DataFrame
df = dataframes[main_record_set_id]

# For this clinical dataset, likely candidates are age, interval, counts. Adjust as needed.
print('Column @id list:', list(df.columns))

# Replace the following field IDs as per your actual dataset schema (they must be @id, not labels):
numeric_field_id = None
group_field_id = None

# Try to detect plausible numeric/group candidates:
for col in df.columns:
    if numeric_field_id is None and np.issubdtype(df[col].dtype, np.number):
        numeric_field_id = col
    if group_field_id is None and ('sex' in col.lower() or 'site' in col.lower() or 'category' in col.lower() or 'group' in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    print("No numeric field found. Please specify the correct field @id.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")

if group_field_id is not None:
    print(f"Using group field @id: {group_field_id}")
else:
    print("No group field detected. Proceeding without grouping.")

# Filtering: Choose threshold based on field range
if numeric_field_id is not None:
    thresh = df[numeric_field_id].quantile(0.75)  # use 75th percentile as sample threshold
    filtered_df = df[df[numeric_field_id] > thresh].copy()
    print(f"\nFiltered records with {numeric_field_id} > {thresh}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
        / filtered_df[numeric_field_id].std()
    )

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional grouping
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean").reset_index()
        print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print('Skipping filtering/normalizing as no suitable numeric field found.')

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and if available, by groups. Please adjust or extend visualizations based on actual data fields.

In [ ]:
if numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].hist(bins=15, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.grid(True, axis='y', alpha=0.2)
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

We have loaded and explored the FAIR² colorectal cancer survivor dataset using `mlcroissant`, referencing all Croissant entities by their `@id`. You can now further analyze relationships, test clinical hypotheses, or prepare the dataset for statistical or machine learning tasks.

Consider extending this notebook with:
- Additional feature engineering or variable transformations referencing precise field `@id`s
- Integration with ML workflows using pandas DataFrames
- More advanced visualizations or statistical testing as appropriate

**Remember:** When referencing record sets, fields, and columns, always use the Croissant `@id` for reproducibility and schema consistency.